# Exploração Inicial do Limit Order Book (LOB)

Este notebook serve como ponto de partida para:
- carregar dados da B3 (ex.: CSV em `../data/`)
- inspecionar o livro de ofertas (LOB)
- preparar features para o ambiente `B3LimitOrderBookEnv` e para a rede MoE.

In [9]:
# Configuração de caminho para importar o pacote local `src`
import sys
from pathlib import Path

ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

In [21]:
# Importações padrão de Data Science
import pandas as pd
import numpy as np
import torch

# Importando as nossas classes do diretório src
from src.sor_env import B3LimitOrderBookEnv
from src.moe_dqn import Expert as MoENetwork

# Configurando o PyTorch para não usar notação científica (facilita a leitura)
torch.set_printoptions(sci_mode=False, precision=4)

### Criação dos Dados Simulados (LOB)

In [11]:
# Criando um DataFrame simulando 10 milissegundos do Limit Order Book da B3
dados_mock = {
    'bid': [35.00, 35.01, 35.02, 34.99, 35.05, 35.04, 35.06, 35.03, 35.07, 35.08],
    'ask': [35.02, 35.03, 35.05, 35.01, 35.06, 35.05, 35.08, 35.04, 35.09, 35.10],
    'volume': [1500, 2000, 800, 3000, 500, 1200, 400, 2500, 300, 100]
}
df_b3 = pd.DataFrame(dados_mock)

print("Visualizando os primeiros ticks do LOB:")
display(df_b3.head())

Visualizando os primeiros ticks do LOB:


,bid,ask,volume
0,35.00,35.02,1500
1,35.01,35.03,2000
2,35.02,35.05,800
3,34.99,35.01,3000
4,35.05,35.06,500


### Instanciando o Ambiente

In [14]:
# Inicializa o ambiente com uma ordem institucional de 5.000 ações
env = B3LimitOrderBookEnv(df_dados=df_b3, total_inventory=5000)

# Dá o reset para capturar o primeiro estado do mercado
estado_atual, info = env.reset()

print("--- ESTADO INICIAL DO AMBIENTE ---")
print(f"Vetor de Estado [Bid, Ask, Vol_Bid, Vol_Ask, Inventário]: {estado_atual}")
print(f"Inventário Restante: {info['inventory_left']} ações")
print(f"Preço de Chegada (Benchmark): R$ {env.arrival_price:.2f}")

--- ESTADO INICIAL DO AMBIENTE ---
Vetor de Estado [Bid, Ask, Vol_Bid, Vol_Ask, Inventário]: [  35.     35.02 1500.   1500.   5000.  ]
Inventário Restante: 5000 ações
Preço de Chegada (Benchmark): R$ 35.01


### Instanciando a Inteligência Artificial (MoE)

In [ ]:
# Inicializa a rede neural Mixture of Experts
# 5 entradas (tamanho do estado), 3 saídas (ações possíveis), 3 especialistas
agente_moe = MoENetwork(input_dim=5, output_dim=3, num_experts=3)

print("--- ARQUITETURA DA REDE MoE ---")
print(agente_moe)

--- ARQUITETURA DA REDE MoE ---
Expert(
  (gating_network): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=3, bias=True)
  )
  (experts): ModuleList(
    (0-2): 3 x SingleExpert(
      (network): Sequential(
        (0): Linear(in_features=5, out_features=64, bias=True)
        (1): ReLU()
        (2): Linear(in_features=64, out_features=64, bias=True)
        (3): ReLU()
        (4): Linear(in_features=64, out_features=3, bias=True)
      )
    )
  )
)


### A Mágica da Integração (O Agente "Pensa")

In [17]:
# 1. O PyTorch exige que o estado seja um Tensor. 
# Usamos unsqueeze(0) para simular um "lote" (batch) de 1 única linha.
estado_tensor = torch.tensor(estado_atual, dtype=torch.float32).unsqueeze(0)

# 2. Passamos o estado do mercado pela Rede Neural (Forward Pass)
# Usamos torch.no_grad() porque não estamos treinando a rede agora, apenas usando para prever.
with torch.no_grad():
    q_values = agente_moe(estado_tensor)

    # A rede de chaveamento (Gating Network) decide os pesos dos especialistas
    pesos_especialistas = torch.softmax(agente_moe.gating_network(estado_tensor), dim=-1)

print("--- O CÉREBRO DO AGENTE EM AÇÃO ---")
print(f"Pesos dados pelo Gerente aos 3 Especialistas: {pesos_especialistas.numpy()[0]}")
print(f"Q-Values gerados (Pontuação para Ação 0, 1 e 2): {q_values.numpy()[0]}")

# 3. O agente escolhe a ação com a maior pontuação (maior Q-Value)
acao_escolhida = torch.argmax(q_values).item()

mapa_acoes = {0: "Esperar", 1: "Comprar Lote Pequeno (100)", 2: "Comprar Lote Grande (500)"}
print(f"\n=> Decisão da IA: Ação {acao_escolhida} ({mapa_acoes[acao_escolhida]})")

--- O CÉREBRO DO AGENTE EM AÇÃO ---
Pesos dados pelo Gerente aos 3 Especialistas: [1. 0. 0.]
Q-Values gerados (Pontuação para Ação 0, 1 e 2): [166.058   263.8848  477.97223]

=> Decisão da IA: Ação 2 (Comprar Lote Grande (500))


### Executando a Ação no Mercado

In [18]:
# 4. Enviamos a decisão da IA de volta para o ambiente da B3
proximo_estado, recompensa, terminado, truncado, info_step = env.step(acao_escolhida)

print("--- RESULTADO DA EXECUÇÃO ---")
print(f"Preço de Execução: R$ {info_step['execution_price']:.2f}")
print(f"Custo/Recompensa (Slippage): {recompensa:.4f}")
print(f"Inventário Restante: {info_step['inventory_left']} ações")
print(f"Novo Estado do Mercado: {proximo_estado}")

--- RESULTADO DA EXECUÇÃO ---
Preço de Execução: R$ 35.02
Custo/Recompensa (Slippage): -5.0000
Inventário Restante: 4500 ações
Novo Estado do Mercado: [  35.01   35.03 2000.   2000.   4500.  ]
